# Diablo decision engine: fine-tune Laya typed-decisions (Kaggle, GPU T4 x2)

Trains `convaiinnovations/laya` / `typed-decisions` (Laya 0.3.20, pinned snapshot) on the Diablo decision
dataset v1 with upstream's RLCD recipe, Diablo's weighting, validation-based model selection and
calibration on Diablo's own calibration split.

**Settings:** Accelerator `GPU T4 x2`, Internet `On`.
**Input:** the Kaggle Dataset `diablo-laya`, built on the Mac by `research/laya_diablo/kaggle/bundle.py`.
It holds `contract.json`, `configs/`, `train/` and only the train, validation and calibration splits.
**Output:** `/kaggle/working/laya-diablo-decision-v1.zip`: download it to the Mac.

The held-out test and private splits are never uploaded, and this notebook refuses to run if it finds
them. Model selection uses validation only; the test splits are scored on the Mac with
`research/laya_diablo/eval/after_training.sh`, against Diablo's rules on the same examples.

In [ ]:
!nvidia-smi
import torch, os
n = torch.cuda.device_count()
print("GPUs:", n, [torch.cuda.get_device_name(i) for i in range(n)])
assert n >= 1, "Select GPU T4 x2 under Notebook options -> Accelerator"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
!pip install -q "laya==0.3.20" "transformers>=4.48.0" safetensors huggingface_hub
import laya, transformers
print("laya", laya.__version__, "transformers", transformers.__version__,
      "torch", torch.__version__, "cuda", torch.version.cuda)
assert laya.__version__ == "0.3.20", "the manifest records the Laya version; pin it"

In [ ]:
import shutil, json, hashlib, glob
found = glob.glob("/kaggle/input/**/contract.json", recursive=True)
assert len(found) == 1, f"attach exactly one diablo-laya dataset (found {found})"
SRC = os.path.dirname(found[0])
WORK = "/kaggle/working/laya_diablo"
shutil.copytree(SRC, WORK, dirs_exist_ok=True)
held_out = [p for p in glob.glob(f"{WORK}/**/*.jsonl", recursive=True)
            if os.path.basename(p) in ("test.jsonl", "test_private.jsonl")]
assert not held_out, f"held-out splits must never reach the training machine: {held_out}"
manifest = json.load(open(f"{WORK}/dataset/v1/manifest.json"))
for split in ("train", "validation", "calibration"):
    digest = hashlib.sha256(open(f"{WORK}/dataset/v1/{split}.jsonl", "rb").read()).hexdigest()
    assert digest == manifest["split_hashes"][split], f"{split} does not match the dataset manifest"
CFG = json.load(open(f"{WORK}/configs/train_v1.json"))
print("dataset v1 verified:", {k: manifest["counts"][k] for k in ("train", "validation", "calibration")})

In [ ]:
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
root = snapshot_download(CFG["base"]["repo"], revision=CFG["base"]["revision"],
                         allow_patterns=[f"{CFG['base']['subfolder']}/*"])
assert CFG["base"]["revision"] in root, f"not the pinned base snapshot: {root}"
BASE = f"{root}/{CFG['base']['subfolder']}"
_fix_tokenizer_config(BASE)
print(json.load(open(f"{BASE}/rl_agent_config.json"))["model_name"], "at", CFG["base"]["revision"])

In [ ]:
%cd /kaggle/working/laya_diablo/train
!python prepare_items.py --model-dir {BASE} --config ../configs/train_v1.json --out /kaggle/working/items 2>&1 | tee /kaggle/working/prepare.log

In [ ]:
import time
t0 = time.time()
!torchrun --standalone --nproc_per_node={max(1, torch.cuda.device_count())} train_diablo.py \
    --model-dir {BASE} --items /kaggle/working/items --out /kaggle/working/laya-diablo-decision-v1 \
    --config ../configs/train_v1.json --dataset-manifest ../dataset/v1/manifest.json \
    2>&1 | tee /kaggle/working/train.log
print("training minutes", round((time.time() - t0) / 60, 1))

In [ ]:
out = "/kaggle/working/laya-diablo-decision-v1"
m = json.load(open(f"{out}/diablo_manifest.json"))
print(json.dumps(m["training"]["validation_by_epoch"], indent=1))
print("run", m["run_id"], "steps", m["training"]["optimizer_steps"], "best epoch", m["training"]["best_epoch"])
print("calibration", json.dumps(m["calibration"]))
print("environment", m["environment"])
assert m["training"]["optimizer_steps"] > 0 and not m["training"]["dry_run"]
assert "temperature_by_options" not in json.load(open(f"{out}/rl_agent_config.json"))
for log in ("prepare.log", "train.log"):
    shutil.copy(f"/kaggle/working/{log}", f"{out}/{log}")
shutil.make_archive(out, "zip", out)
print("download:", out + ".zip")

## On the Mac

    unzip laya-diablo-decision-v1.zip -d research/laya_diablo/checkpoints/laya-diablo-decision-v1
    research/laya_diablo/eval/after_training.sh research/laya_diablo/checkpoints/laya-diablo-decision-v1

It refuses a checkpoint that fails `verify_checkpoint` (contract, questions, base snapshot, dataset hashes,
calibration, weights), then scores Rules, the checkpoint and Hybrid on the untouched v1 test, runs the gate,
and only then the private test. Rules stay authoritative whatever it says.